# Part 4: Data Subset Analysis

Exploring baseline melanoma `PBMC` samples from patients treated with `miraclib`.

Note: `quintazide` is mentioned here for completeness because it may appear as another treatment in the broader dataset, but this Part 4 subset focuses on `miraclib` as requested.

## Questions

For melanoma PBMC samples at baseline (`time_from_treatment_start = 0`) from patients treated with `miraclib`, determine:

1. How many samples come from each project.
2. How many subjects were responders and non-responders.
3. How many subjects were male and female.
4. Among melanoma males in this same baseline subset, what is the average number of B cells for responders at time 0?

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

DB_PATH = Path("cell_counts.db")

if not DB_PATH.exists():
    raise FileNotFoundError("Run `python load_data.py` before opening this notebook.")

## Load Baseline Melanoma PBMC Miraclib Samples

The query below pulls one row per sample, including subject metadata and the B-cell count needed for the final question.

In [2]:
baseline_query = """
SELECT
    samples.sample_id AS sample,
    subjects.subject_id,
    subjects.project,
    subjects.indication,
    subjects.gender,
    samples.treatment,
    samples.sample_type,
    samples.response,
    samples.time_from_treatment_start,
    counts.cell_count AS b_cell_count
FROM samples
INNER JOIN subjects
    ON subjects.subject_id = samples.subject_id
INNER JOIN cell_counts AS counts
    ON counts.sample_id = samples.sample_id
INNER JOIN cell_populations AS populations
    ON populations.population_id = counts.population_id
WHERE LOWER(subjects.indication) = 'melanoma'
    AND UPPER(samples.sample_type) = 'PBMC'
    AND LOWER(samples.treatment) = 'miraclib'
    AND samples.time_from_treatment_start = 0
    AND populations.population_name = 'b_cell';
"""

with sqlite3.connect(DB_PATH) as connection:
    baseline_df = pd.read_sql_query(baseline_query, connection)

baseline_df.head()

,sample,subject_id,project,indication,gender,treatment,sample_type,response,time_from_treatment_start,b_cell_count
0,sample00000,sbj000,prj1,melanoma,M,miraclib,PBMC,no,0,10908
1,sample00006,sbj002,prj1,melanoma,M,miraclib,PBMC,no,0,5897
2,sample00009,sbj003,prj1,melanoma,M,miraclib,PBMC,no,0,7650
3,sample00024,sbj008,prj1,melanoma,F,miraclib,PBMC,yes,0,7552
4,sample00060,sbj020,prj1,melanoma,F,miraclib,PBMC,yes,0,11035


## Subset Size

Confirm how many baseline samples and unique subjects are included in the subset.

In [3]:
subset_summary = pd.DataFrame(
    {
        "metric": ["samples", "subjects"],
        "count": [baseline_df["sample"].nunique(), baseline_df["subject_id"].nunique()],
    }
)

subset_summary

,metric,count
0,samples,656
1,subjects,656


## Samples From Each Project

Count baseline samples by project.

In [4]:
samples_by_project = (
    baseline_df.groupby("project", as_index=False)["sample"]
    .nunique()
    .rename(columns={"sample": "sample_count"})
    .sort_values("project")
)

samples_by_project

,project,sample_count
0,prj1,384
1,prj3,272


## Subjects by Response

Count unique subjects by responder status.

In [5]:
subjects_by_response = (
    baseline_df.groupby("response", as_index=False)["subject_id"]
    .nunique()
    .rename(columns={"subject_id": "subject_count"})
    .sort_values("response")
)

subjects_by_response

,response,subject_count
0,no,325
1,yes,331


## Subjects by Gender

Count unique subjects by gender.

In [6]:
subjects_by_gender = (
    baseline_df.groupby("gender", as_index=False)["subject_id"]
    .nunique()
    .rename(columns={"subject_id": "subject_count"})
    .sort_values("gender")
)

subjects_by_gender

,gender,subject_count
0,F,312
1,M,344


## Average B Cells for Male Responders at Baseline

Using the same melanoma PBMC miraclib baseline subset, calculate the average B-cell count for male responders at time 0. The result is rounded to two decimal places.

In [7]:
male_responder_b_cells = baseline_df[
    (baseline_df["gender"].str.upper() == "M")
    & (baseline_df["response"].str.lower() == "yes")
]["b_cell_count"]

average_b_cells_male_responders = round(male_responder_b_cells.mean(), 2)

average_b_cells_male_responders

10401.28